# 03 — Data Integration

## Purpose

This notebook integrates the cleaned datasets prepared during the data-cleaning stage of the PMIP project.

The project currently contains three cleaned datasets:

1. **Listeners Dataset** — listener and music-consumption information.
2. **Spotify 2024 Dataset** — track-level streaming and cross-platform performance metrics.
3. **Charts Dataset** — historical chart performance across countries and dates.

Before combining any datasets, their structures and potential relationships will be examined carefully. This is important because datasets should only be merged when suitable common attributes can be identified.

## Objectives

The main objectives of this notebook are to:

- Load the cleaned datasets from `data/processed/`.
- Inspect their dimensions and column structures.
- Identify possible relationships and common fields between datasets.
- Determine appropriate keys for integration.
- Integrate compatible datasets without introducing unnecessary duplicates or data loss.
- Validate the resulting integrated data.
- Save the prepared dataset for the next stage of the project.

## Table of Contents

1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Identify Processed Datasets](#2-identify-processed-datasets)
3. [Load the Cleaned Datasets](#3-load-the-cleaned-datasets)
4. [Dataset Structure Comparison](#4-dataset-structure-comparison)
5. [Potential Integration Fields](#5-potential-integration-fields)
6. [Integration Field Mapping](#6-integration-field-mapping)
7. [Candidate Integration Key Inspection](#7-candidate-integration-key-inspection)
8. [Track and Artist Overlap Analysis](#8-track-and-artist-overlap-analysis)
9. [Composite Track Matching Analysis](#9-composite-track-matching-analysis)
   - [9.1 Track and Artist Combined Matching](#91-track-and-artist-combined-matching)
   - [9.2 Historical Chart Coverage](#92-historical-chart-coverage-of-matched-tracks)
   - [9.3 Match Relationship Validation](#93-match-relationship-validation)
   - [9.4 Historical Chart Feature Aggregation](#94-historical-chart-feature-aggregation)
   - [9.5 Memory-Efficient Historical Chart Aggregation](#95-memory-efficient-historical-chart-aggregation)
   - [9.6 Historical Feature Aggregation](#96-historical-feature-aggregation)
   - [9.7 Historical Feature Validation](#97-historical-feature-validation)
10. [Integration with Spotify 2024](#10-integration-with-spotify-2024)
   - [10.1 Inspect Spotify Integration Fields](#101-inspect-spotify-integration-fields)
   - [10.2 Create Spotify Integration Keys](#102-create-spotify-integration-keys)
   - [10.3 Validate Historical Feature Matches](#103-validate-historical-feature-matches)
   - [10.4 Integrate Historical Chart Features](#104-integrate-historical-chart-features)
   - [10.5 Prepare Artist Listener Integration](#105-prepare-artist-listener-integration)
   - [10.6 Validate Spotify and Listener Artist Matches](#106-validate-spotify-and-listener-artist-matches)
   - [10.7 Integrate Artist Listener Features](#107-integrate-artist-listener-features)
   - [10.8 Final Integrated Dataset Validation](#108-final-integrated-dataset-validation)
   - [10.9 Save the Integrated Dataset](#109-save-the-integrated-dataset)
   - [10.10 Reload and Verify the Saved Dataset](#1010-reload-and-verify-the-saved-dataset)
11. [Integration Summary](#11-integration-summary)


## 1. Setup and Data Loading

The required Python libraries are imported first. The cleaned datasets produced during the previous data-cleaning stage will then be loaded from the `data/processed/` directory.


In [14]:
import pandas as pd
from pathlib import Path

# Path to processed datasets
processed_data_path = Path("../data/processed")

print("Setup completed successfully.")

Setup completed successfully.


## 2. Identify Processed Datasets

Before loading the cleaned datasets, the contents of the `data/processed/` directory are inspected.

This confirms which datasets were successfully produced during the data-cleaning stage and ensures that the integration process uses the cleaned versions rather than the original raw datasets.


In [15]:
# List all files in the processed data directory

processed_files = list(processed_data_path.iterdir())

print("Files available in data/processed:")
print("=" * 40)

for file in processed_files:
    print(file.name)

print(f"\nTotal processed files: {len(processed_files)}")

Files available in data/processed:
spotify_2024_cleaned.csv
listeners_cleaned.csv
charts_cleaned.csv

Total processed files: 3


## 3. Load the Cleaned Datasets

All three cleaned datasets are now available in the `data/processed/` directory.

They are loaded into separate Pandas DataFrames so that their structures can be compared before any integration is attempted.

At this stage, no datasets are merged. The aim is only to confirm that each cleaned dataset loads successfully and retains the expected number of rows and columns.


In [16]:
# Load the three cleaned datasets

spotify_df = pd.read_csv(
    processed_data_path / "spotify_2024_cleaned.csv",
    parse_dates=["Release Date"]
)

listeners_df = pd.read_csv(
    processed_data_path / "listeners_cleaned.csv"
)

charts_df = pd.read_csv(
    processed_data_path / "charts_cleaned.csv",
    parse_dates=["date"]
)

print("Datasets loaded successfully.")
print("=" * 40)

print(f"Spotify dataset:   {spotify_df.shape}")
print(f"Listeners dataset: {listeners_df.shape}")
print(f"Charts dataset:    {charts_df.shape}")

Datasets loaded successfully.
Spotify dataset:   (4593, 27)
Listeners dataset: (2500, 5)
Charts dataset:    (5427136, 10)


## 4. Dataset Structure Comparison

Before integrating the datasets, their column structures must be compared.

A successful data integration requires one or more meaningful fields that can connect records between datasets. For example, music datasets may share information such as track names, artist names, track identifiers, dates, or other attributes.

The columns of each dataset are therefore inspected to identify potential relationships and candidate integration keys.


In [17]:
# Display the columns contained in each dataset

print("SPOTIFY DATASET COLUMNS")
print("=" * 50)

for column in spotify_df.columns:
    print(f"- {column}")


print("\nLISTENERS DATASET COLUMNS")
print("=" * 50)

for column in listeners_df.columns:
    print(f"- {column}")


print("\nCHARTS DATASET COLUMNS")
print("=" * 50)

for column in charts_df.columns:
    print(f"- {column}")

SPOTIFY DATASET COLUMNS
- Track
- Album Name
- Artist
- Release Date
- ISRC
- All Time Rank
- Track Score
- Spotify Streams
- Spotify Playlist Count
- Spotify Playlist Reach
- Spotify Popularity
- YouTube Views
- YouTube Likes
- TikTok Posts
- TikTok Likes
- TikTok Views
- YouTube Playlist Reach
- Apple Music Playlist Count
- AirPlay Spins
- SiriusXM Spins
- Deezer Playlist Count
- Deezer Playlist Reach
- Amazon Playlist Count
- Pandora Streams
- Pandora Track Stations
- Shazam Counts
- Explicit Track

LISTENERS DATASET COLUMNS
- Artist
- Listeners
- Daily Trend
- Peak
- PkListeners

CHARTS DATASET COLUMNS
- date
- country
- position
- streams
- track_id
- artists
- artist_genres
- duration
- explicit
- name


## 5. Potential Integration Fields

The three datasets use different column names and represent different types of music information. Therefore, similarly named or related fields cannot automatically be treated as integration keys.

The next step is to create a compact comparison of the available columns and identify fields that may represent the same real-world information.

Particular attention will be given to track names, artist names, identifiers and other attributes that could provide reliable links between datasets.


In [18]:
# Create a compact summary of the dataset structures

datasets = {
    "Spotify 2024": spotify_df,
    "Listeners": listeners_df,
    "Charts": charts_df
}

for dataset_name, df in datasets.items():
    print(f"\n{dataset_name.upper()}")
    print("=" * 50)
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print("Column names:")
    
    for column in df.columns:
        print(f"  - {column}")


SPOTIFY 2024
Rows: 4,593
Columns: 27
Column names:
  - Track
  - Album Name
  - Artist
  - Release Date
  - ISRC
  - All Time Rank
  - Track Score
  - Spotify Streams
  - Spotify Playlist Count
  - Spotify Playlist Reach
  - Spotify Popularity
  - YouTube Views
  - YouTube Likes
  - TikTok Posts
  - TikTok Likes
  - TikTok Views
  - YouTube Playlist Reach
  - Apple Music Playlist Count
  - AirPlay Spins
  - SiriusXM Spins
  - Deezer Playlist Count
  - Deezer Playlist Reach
  - Amazon Playlist Count
  - Pandora Streams
  - Pandora Track Stations
  - Shazam Counts
  - Explicit Track

LISTENERS
Rows: 2,500
Columns: 5
Column names:
  - Artist
  - Listeners
  - Daily Trend
  - Peak
  - PkListeners

CHARTS
Rows: 5,427,136
Columns: 10
Column names:
  - date
  - country
  - position
  - streams
  - track_id
  - artists
  - artist_genres
  - duration
  - explicit
  - name


## 6. Integration Field Mapping

The complete column lists are too large to compare efficiently as printed output. Therefore, the analysis is narrowed to fields that are potentially relevant for connecting the datasets.

The purpose of this step is not to merge the datasets yet, but to identify candidate fields that represent similar information across the datasets.

Potential relationships will then be tested using the actual values contained in those fields before any integration decision is made.


In [19]:
# Display the columns of each dataset in a compact table

max_columns = max(
    len(spotify_df.columns),
    len(listeners_df.columns),
    len(charts_df.columns)
)

column_comparison = pd.DataFrame({
    "Spotify 2024": list(spotify_df.columns) + [None] * (max_columns - len(spotify_df.columns)),
    "Listeners": list(listeners_df.columns) + [None] * (max_columns - len(listeners_df.columns)),
    "Charts": list(charts_df.columns) + [None] * (max_columns - len(charts_df.columns))
})

column_comparison

,Spotify 2024,Listeners,Charts
0,Track,Artist,date
1,Album Name,Listeners,country
2,Artist,Daily Trend,position
3,Release Date,Peak,streams
4,ISRC,PkListeners,track_id
5,All Time Rank,NaN,artists
6,Track Score,NaN,artist_genres
7,Spotify Streams,NaN,duration
8,Spotify Playlist Count,NaN,explicit
9,Spotify Playlist Reach,NaN,name


## 7. Candidate Integration Key Inspection

The structural comparison suggests that the datasets operate at different levels of detail.

The Spotify 2024 and Charts datasets primarily contain track-level information, while the Listeners dataset contains artist-level listener statistics.

Potential relationships include:

- `Spotify 2024: Track` ↔ `Charts: name`
- `Spotify 2024: Artist` ↔ `Charts: artists`
- `Listeners: Artist` ↔ `Spotify 2024: Artist`

However, column names alone are not sufficient evidence for integration. The actual values and formatting of these fields must first be inspected.

This step therefore examines representative values from the candidate integration fields before any matching or merging is performed.


In [20]:
# Inspect sample values from potential integration fields

print("SPOTIFY 2024")
print("=" * 50)
print(
    spotify_df[
        ["Track", "Artist", "ISRC"]
    ].head(10).to_string(index=False)
)

print("\nLISTENERS")
print("=" * 50)
print(
    listeners_df[
        ["Artist", "Listeners", "Daily Trend", "Peak", "PkListeners"]
    ].head(10).to_string(index=False)
)

print("\nCHARTS")
print("=" * 50)
print(
    charts_df[
        ["name", "artists", "track_id"]
    ].head(10).to_string(index=False)
)

SPOTIFY 2024
                     Track         Artist         ISRC
       MILLION DOLLAR BABY  Tommy Richman QM24S2402528
               Not Like Us Kendrick Lamar USUG12400910
i like the way you kiss me        Artemas QZJ842400387
                   Flowers    Miley Cyrus USSM12209777
                   Houdini         Eminem USUG12403398
               Lovin On Me    Jack Harlow USAT22311371
          Beautiful Things   Benson Boone USWB12307016
                 Gata Only     FloyyMenor QZL382406049
      Danza Kuduro - Cover  MUSIC LAB JPN TCJPA2463708
BAND4BAND (feat. Lil Baby)    Central Cee USSM12404354

LISTENERS
       Artist  Listeners  Daily Trend  Peak  PkListeners
   The Weeknd  107592328      -138880     1    113034886
 Taylor Swift  101003302          889     2    101003302
   Ed Sheeran   76475126       -68137     2     87934910
     Dua Lipa   76421916       -71356     4     77778397
    Bad Bunny   76162057      -199052     3     83950570
      Rihanna   75784389     

## 8. Track and Artist Overlap Analysis

Before integrating the datasets, the level of overlap between their candidate matching fields must be measured.

The Spotify 2024 and Charts datasets both contain track and artist information, although their column names and formats differ. The Listeners dataset contains artist-level statistics that may potentially be connected to artists appearing in the Spotify dataset.

For this initial comparison, text values are temporarily normalised by converting them to lowercase and removing surrounding whitespace. This does not modify the original dataset columns.

The analysis measures:

- track-name overlap between Spotify 2024 and Charts;
- artist overlap between Spotify 2024 and Listeners;
- the proportion of Spotify records represented in the other datasets.

These results will help determine whether direct integration is sufficiently reliable or whether further standardisation is required.


In [21]:
# Create temporary normalised values for overlap analysis

spotify_track_names = (
    spotify_df["Track"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

chart_track_names = (
    charts_df["name"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

spotify_artists = (
    spotify_df["Artist"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

listener_artists = (
    listeners_df["Artist"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)


# Convert to sets of unique values
spotify_track_set = set(spotify_track_names)
chart_track_set = set(chart_track_names)

spotify_artist_set = set(spotify_artists)
listener_artist_set = set(listener_artists)


# Find overlaps
track_overlap = spotify_track_set.intersection(chart_track_set)
artist_overlap = spotify_artist_set.intersection(listener_artist_set)


# Calculate percentages
track_overlap_percentage = (
    len(track_overlap) / len(spotify_track_set) * 100
)

artist_overlap_percentage = (
    len(artist_overlap) / len(spotify_artist_set) * 100
)


print("DATASET OVERLAP ANALYSIS")
print("=" * 55)

print("\nSpotify 2024 ↔ Charts")
print("-" * 55)
print(f"Unique Spotify track names: {len(spotify_track_set):,}")
print(f"Unique Charts track names:  {len(chart_track_set):,}")
print(f"Matching track names:       {len(track_overlap):,}")
print(f"Spotify track overlap:      {track_overlap_percentage:.2f}%")

print("\nSpotify 2024 ↔ Listeners")
print("-" * 55)
print(f"Unique Spotify artists:     {len(spotify_artist_set):,}")
print(f"Unique Listener artists:    {len(listener_artist_set):,}")
print(f"Matching artists:           {len(artist_overlap):,}")
print(f"Spotify artist overlap:     {artist_overlap_percentage:.2f}%")

DATASET OVERLAP ANALYSIS

Spotify 2024 ↔ Charts
-------------------------------------------------------
Unique Spotify track names: 4,314
Unique Charts track names:  93,470
Matching track names:       2,499
Spotify track overlap:      57.93%

Spotify 2024 ↔ Listeners
-------------------------------------------------------
Unique Spotify artists:     1,997
Unique Listener artists:    2,500
Matching artists:           1,003
Spotify artist overlap:     50.23%


## 9. Composite Track Matching Analysis

The initial overlap analysis identified a substantial number of shared track names between the Spotify 2024 and Charts datasets. However, track titles alone are not sufficiently reliable for integration because different artists may release tracks with identical or similar names.

A stronger matching strategy is therefore required using both the track title and artist information.

The Spotify dataset stores the primary artist as a text value, while the Charts dataset stores artists as string representations of lists. The Charts artist field must therefore be parsed before the datasets can be compared reliably.

This stage creates temporary normalised track and artist values and measures how many Spotify records can be matched to the Charts dataset using both attributes.


In [22]:
import ast

# Create Spotify matching table
spotify_match = spotify_df[
    ["Track", "Artist"]
].copy()

spotify_match["track_normalised"] = (
    spotify_match["Track"]
    .astype(str)
    .str.strip()
    .str.lower()
)

spotify_match["artist_normalised"] = (
    spotify_match["Artist"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# IMPORTANT:
# Use only one record per Charts track_id for matching
charts_catalog = (
    charts_df[
        ["track_id", "name", "artists"]
    ]
    .drop_duplicates(subset="track_id")
    .copy()
)

charts_catalog["track_normalised"] = (
    charts_catalog["name"]
    .astype(str)
    .str.strip()
    .str.lower()
)


def parse_artists(value):
    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [
                str(artist).strip().lower()
                for artist in parsed
            ]

    except (ValueError, SyntaxError, TypeError):
        pass

    return []


charts_catalog["artist_list"] = (
    charts_catalog["artists"]
    .apply(parse_artists)
)

print("Unique Charts tracks prepared:", f"{len(charts_catalog):,}")

Unique Charts tracks prepared: 110,198


### 9.1 Track and Artist Combined Matching

Track-title overlap alone does not guarantee that two records represent the same song, as different artists may use identical track titles. To improve the reliability of the integration, both track title and artist information are considered together.

For each Charts record, the parsed artist list is expanded so that each artist can be compared individually with the Spotify artist field. A record is considered a potential match when both the normalised track title and normalised artist name agree.

This provides a more reliable estimate of the number of Spotify tracks that can be linked to historical chart observations.


In [23]:
# Expand artists only for the unique Charts track catalogue
charts_artist_expanded = (
    charts_catalog[
        ["track_id", "track_normalised", "artist_list"]
    ]
    .explode("artist_list")
    .rename(columns={"artist_list": "artist_normalised"})
)

charts_artist_expanded = charts_artist_expanded[
    charts_artist_expanded["artist_normalised"].notna()
].copy()


spotify_pairs = (
    spotify_match[
        ["track_normalised", "artist_normalised"]
    ]
    .drop_duplicates()
)

chart_pairs = (
    charts_artist_expanded[
        ["track_id", "track_normalised", "artist_normalised"]
    ]
    .drop_duplicates()
)


# Keep track_id because we will need it later
combined_matches = spotify_pairs.merge(
    chart_pairs,
    on=["track_normalised", "artist_normalised"],
    how="inner"
)


matched_spotify_pairs = (
    combined_matches[
        ["track_normalised", "artist_normalised"]
    ]
    .drop_duplicates()
)

combined_match_percentage = (
    len(matched_spotify_pairs) /
    len(spotify_pairs) *
    100
)


print("TRACK + ARTIST MATCHING ANALYSIS")
print("=" * 55)

print(
    f"\nUnique Spotify track-artist pairs: "
    f"{len(spotify_pairs):,}"
)

print(
    f"Matched Spotify track-artist pairs: "
    f"{len(matched_spotify_pairs):,}"
)

print(
    f"Spotify combined match rate: "
    f"{combined_match_percentage:.2f}%"
)

print(
    f"Matched Charts track IDs: "
    f"{combined_matches['track_id'].nunique():,}"
)

TRACK + ARTIST MATCHING ANALYSIS

Unique Spotify track-artist pairs: 4,480
Matched Spotify track-artist pairs: 2,178
Spotify combined match rate: 48.62%
Matched Charts track IDs: 2,268


### 9.2 Historical Chart Coverage of Matched Tracks

The track-and-artist matching stage identified the Charts track IDs corresponding to reliably matched Spotify records.

Because the Charts dataset contains more than 5.4 million historical observations, performing repeated large joins would use unnecessary memory. Instead, the matched track IDs are used to filter the original Charts dataset directly.

This preserves the full historical observations for matched tracks while avoiding unnecessary duplication during integration.


In [24]:
# Get Charts track IDs belonging to reliable Spotify matches
matched_track_ids = combined_matches["track_id"].unique()

# Filter instead of merging millions of records
matched_chart_history = charts_df[
    charts_df["track_id"].isin(matched_track_ids)
].copy()


print("MATCHED HISTORICAL CHART COVERAGE")
print("=" * 60)

print(
    f"\nMatched Charts track IDs: "
    f"{len(matched_track_ids):,}"
)

print(
    f"Historical chart observations: "
    f"{len(matched_chart_history):,}"
)

print(
    f"Countries represented: "
    f"{matched_chart_history['country'].nunique():,}"
)

print(
    f"Earliest chart observation: "
    f"{matched_chart_history['date'].min()}"
)

print(
    f"Latest chart observation: "
    f"{matched_chart_history['date'].max()}"
)

MATCHED HISTORICAL CHART COVERAGE

Matched Charts track IDs: 2,268
Historical chart observations: 2,113,484
Countries represented: 77
Earliest chart observation: 2013-04-28 00:00:00
Latest chart observation: 2023-04-06 00:00:00


### 9.3 Match Relationship Validation

The matching process identified more Charts track IDs than Spotify track-and-artist pairs. This suggests that some Spotify tracks may correspond to multiple track identifiers in the historical Charts dataset.

This is not automatically an error, as different versions or releases of the same track may use separate track IDs.

Before aggregating the historical chart data, the relationship between Spotify track-artist pairs and Charts track IDs is examined. This helps ensure that the final integration maintains one record per Spotify track rather than creating duplicate observations.


In [25]:
# Inspect the relationship between matched Spotify pairs and Charts track IDs

match_relationships = combined_matches[
    [
        "track_normalised",
        "artist_normalised",
        "track_id"
    ]
].drop_duplicates()


# Number of Charts IDs associated with each Spotify track-artist pair
ids_per_spotify_pair = (
    match_relationships
    .groupby(
        ["track_normalised", "artist_normalised"]
    )["track_id"]
    .nunique()
)


# Number of Spotify pairs associated with each Charts track ID
pairs_per_track_id = (
    match_relationships
    .groupby("track_id")
    .size()
)


print("MATCH RELATIONSHIP VALIDATION")
print("=" * 60)

print(
    "\nSpotify track-artist pairs with one Charts track ID:",
    (ids_per_spotify_pair == 1).sum()
)

print(
    "Spotify track-artist pairs with multiple Charts track IDs:",
    (ids_per_spotify_pair > 1).sum()
)

print(
    "\nMaximum Charts track IDs linked to one Spotify pair:",
    ids_per_spotify_pair.max()
)

print(
    "\nCharts track IDs linked to multiple Spotify pairs:",
    (pairs_per_track_id > 1).sum()
)

MATCH RELATIONSHIP VALIDATION

Spotify track-artist pairs with one Charts track ID: 2093
Spotify track-artist pairs with multiple Charts track IDs: 85

Maximum Charts track IDs linked to one Spotify pair: 3

Charts track IDs linked to multiple Spotify pairs: 0


### 9.4 Historical Chart Feature Aggregation

The matched historical Charts data contains more than two million observations. These records must be summarised before integration because directly merging them with the Spotify dataset would create many repeated rows for the same track.

The normalised track and artist fields are stored in the smaller matching table rather than in the full historical Charts dataset. To avoid unnecessarily copying large text columns across millions of rows, each matched Spotify track-artist pair is assigned a temporary numeric identifier.

The historical observations are then linked to this numeric identifier using `track_id` and aggregated to one record per matched Spotify track-artist pair.

The resulting historical features include:

- best historical chart position;
- average historical chart position;
- number of historical chart observations;
- number of countries charted;
- total historical streams;
- average historical streams;
- maximum historical streams;
- first chart appearance; and
- last chart appearance.


In [27]:
# Create one row for each matched Spotify track-artist pair
pair_lookup = (
    combined_matches[
        ["track_normalised", "artist_normalised"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Give each pair a small numeric ID
pair_lookup["pair_id"] = pair_lookup.index


# Connect each Charts track_id to its Spotify pair
track_id_mapping = (
    combined_matches[
        [
            "track_id",
            "track_normalised",
            "artist_normalised"
        ]
    ]
    .drop_duplicates()
    .merge(
        pair_lookup,
        on=["track_normalised", "artist_normalised"],
        how="left"
    )[
        ["track_id", "pair_id"]
    ]
)


print("Pair mapping prepared.")
print(f"Spotify track-artist pairs: {len(pair_lookup):,}")
print(f"Matched Charts track IDs:   {track_id_mapping['track_id'].nunique():,}")

Pair mapping prepared.
Spotify track-artist pairs: 2,178
Matched Charts track IDs:   2,268


### 9.5 Memory-Efficient Historical Chart Aggregation

The previous aggregation attempt could not directly group the historical chart data by `track_normalised` and `artist_normalised` because those temporary matching fields were not stored in the full historical Charts dataset.

The Charts dataset also contains more than two million matched historical observations, so repeatedly adding long text fields to every row would use unnecessary memory.

To make the integration more efficient, each matched Spotify track-artist pair is assigned a temporary numeric `pair_id`. Each matched Charts `track_id` is then linked to the relevant `pair_id`.

This allows the historical chart observations to be aggregated efficiently while still preserving the relationship back to the original Spotify track and artist.

The temporary numeric identifier is used only during processing and will not be retained in the final integrated dataset.


In [28]:
# Attach only the numeric pair ID to the historical observations
history_with_pair = matched_chart_history.merge(
    track_id_mapping,
    on="track_id",
    how="inner"
)

print("Historical observations linked successfully.")
print(f"Rows: {len(history_with_pair):,}")

history_with_pair[
    [
        "pair_id",
        "date",
        "country",
        "position",
        "streams",
        "track_id"
    ]
].head()

Historical observations linked successfully.
Rows: 2,113,484


,pair_id,date,country,position,streams,track_id
0,190,2013-09-29,global,3,6816696,0nrRP2bk19rLc0orkWPQk2
1,190,2013-10-06,global,3,6532360,0nrRP2bk19rLc0orkWPQk2
2,190,2013-10-13,global,4,6159416,0nrRP2bk19rLc0orkWPQk2
3,190,2013-10-20,global,4,5833226,0nrRP2bk19rLc0orkWPQk2
4,190,2013-10-27,global,5,5571891,0nrRP2bk19rLc0orkWPQk2


### 9.6 Historical Feature Aggregation

The successfully linked historical observations are now aggregated from individual chart records into one summary record for each matched Spotify track-artist pair.

This reduces more than two million historical chart observations into a compact set of historical performance features suitable for integration with the Spotify dataset.

For each matched track-artist pair, the aggregation calculates its best and average chart position, number of chart observations, number of countries reached, historical streaming statistics, and the first and last dates on which the track appeared in the Charts dataset.


In [29]:
# Aggregate historical chart observations by Spotify track-artist pair

historical_features = (
    history_with_pair
    .groupby(
        "pair_id",
        as_index=False
    )
    .agg(
        best_chart_position=("position", "min"),
        average_chart_position=("position", "mean"),
        chart_observations=("position", "size"),
        countries_charted=("country", "nunique"),
        total_historical_streams=("streams", "sum"),
        average_historical_streams=("streams", "mean"),
        max_historical_streams=("streams", "max"),
        first_chart_date=("date", "min"),
        last_chart_date=("date", "max")
    )
)

# Restore track and artist information
historical_features = pair_lookup.merge(
    historical_features,
    on="pair_id",
    how="inner"
)

# pair_id was only required during processing
historical_features = historical_features.drop(
    columns=["pair_id"]
)

# Round averages for readability
historical_features["average_chart_position"] = (
    historical_features["average_chart_position"].round(2)
)

historical_features["average_historical_streams"] = (
    historical_features["average_historical_streams"].round(2)
)

print("HISTORICAL CHART FEATURE AGGREGATION")
print("=" * 60)

print(
    f"Historical observations used: "
    f"{len(history_with_pair):,}"
)

print(
    f"Spotify track-artist records produced: "
    f"{len(historical_features):,}"
)

print(
    f"Historical features produced: "
    f"{historical_features.shape[1] - 2}"
)

print("\nExample aggregated records:")
historical_features.head(10)

HISTORICAL CHART FEATURE AGGREGATION
Historical observations used: 2,113,484
Spotify track-artist records produced: 2,178
Historical features produced: 9

Example aggregated records:


,track_normalised,artist_normalised,best_chart_position,average_chart_position,chart_observations,countries_charted,total_historical_streams,average_historical_streams,max_historical_streams,first_chart_date,last_chart_date
0,flowers,miley cyrus,1,19.70,873,74,1658244203,1899477.90,115156896,2023-01-19,2023-04-06
1,lala,myke towers,168,168.00,1,1,428761,428761.00,428761,2023-03-30,2023-03-30
2,as it was,harry styles,1,21.59,2125,69,2876810895,1353793.36,78460903,2022-04-07,2022-11-10
3,stay (with justin bieber),the kid laroi,1,45.06,4486,70,4132903676,921289.27,70502410,2021-07-15,2022-11-10
4,baby shark,pinkfong,88,162.90,41,6,4331073,105635.93,435691,2017-09-28,2019-02-21
5,numb / encore,jay-z,53,155.74,167,42,16601255,99408.71,5602548,2014-10-26,2018-02-08
6,dance monkey,tones and i,1,66.36,7013,72,4787862859,682712.51,52055226,2019-05-16,2022-11-03
7,i'm good (blue),david guetta,1,48.37,638,65,637152716,998671.97,34345014,2022-09-01,2022-11-10
8,if we ever broke up,mae stephens,15,106.33,128,23,83619615,653278.24,8629147,2023-02-16,2023-04-06
9,despacito,luis fonsi,154,176.33,6,5,692692,115448.67,202179,2019-02-07,2022-10-27


### 9.7 Historical Feature Validation

Before integrating the aggregated historical features with the Spotify 2024 dataset, the resulting values are validated for consistency and plausibility.

The validation checks the number and uniqueness of aggregated track-artist records, missing values, chart position ranges, country coverage, stream values, and historical date ranges.

These checks help identify possible errors introduced during matching or aggregation and ensure that the historical features are suitable for the final integration stage.


In [30]:
# Validate the aggregated historical chart features

print("HISTORICAL FEATURE VALIDATION")
print("=" * 60)

# Basic structure
print(f"\nRows: {len(historical_features):,}")

print(
    "Duplicate track-artist pairs:",
    historical_features.duplicated(
        subset=["track_normalised", "artist_normalised"]
    ).sum()
)

print(
    "Total missing values:",
    historical_features.isna().sum().sum()
)


# Chart position validation
print("\nCHART POSITION")
print("-" * 60)

print(
    "Best position range:",
    historical_features["best_chart_position"].min(),
    "to",
    historical_features["best_chart_position"].max()
)

print(
    "Average position range:",
    historical_features["average_chart_position"].min(),
    "to",
    historical_features["average_chart_position"].max()
)


# Historical observation validation
print("\nCHART COVERAGE")
print("-" * 60)

print(
    "Chart observations range:",
    historical_features["chart_observations"].min(),
    "to",
    historical_features["chart_observations"].max()
)

print(
    "Countries charted range:",
    historical_features["countries_charted"].min(),
    "to",
    historical_features["countries_charted"].max()
)


# Stream validation
print("\nSTREAM FEATURES")
print("-" * 60)

print(
    "Negative total streams:",
    (historical_features["total_historical_streams"] < 0).sum()
)

print(
    "Negative average streams:",
    (historical_features["average_historical_streams"] < 0).sum()
)

print(
    "Negative maximum streams:",
    (historical_features["max_historical_streams"] < 0).sum()
)


# Date validation
print("\nHISTORICAL DATES")
print("-" * 60)

print(
    "Earliest first chart date:",
    historical_features["first_chart_date"].min()
)

print(
    "Latest last chart date:",
    historical_features["last_chart_date"].max()
)

print(
    "Records where first date is after last date:",
    (
        historical_features["first_chart_date"]
        >
        historical_features["last_chart_date"]
    ).sum()
)

HISTORICAL FEATURE VALIDATION

Rows: 2,178
Duplicate track-artist pairs: 0
Total missing values: 0

CHART POSITION
------------------------------------------------------------
Best position range: 1 to 270
Average position range: 6.11 to 270.0

CHART COVERAGE
------------------------------------------------------------
Chart observations range: 1 to 13860
Countries charted range: 1 to 74

STREAM FEATURES
------------------------------------------------------------
Negative total streams: 0
Negative average streams: 0
Negative maximum streams: 0

HISTORICAL DATES
------------------------------------------------------------
Earliest first chart date: 2013-04-28 00:00:00
Latest last chart date: 2023-04-06 00:00:00
Records where first date is after last date: 0


## 10. Integration with Spotify 2024

The validated historical chart features can now be integrated with the Spotify 2024 dataset.

A left join is used so that every record in the Spotify 2024 dataset is retained, including tracks for which no historical Charts match was identified. Tracks with historical matches receive the newly engineered chart-performance features, while unmatched tracks initially contain missing values for those historical attributes.

The normalised track and artist fields used during the matching process act only as temporary integration keys and are not intended to replace the original track and artist information.

### 10.1 Prepare Spotify Integration Keys

Normalised versions of the Spotify track and artist names are prepared using the same matching approach used during the Charts matching stage. These fields provide consistent keys for attaching the aggregated historical features to the original Spotify records.


### 10.1 Inspect Spotify Integration Fields

Before creating new integration keys, the Spotify dataset is inspected to confirm
whether any normalised matching fields already exist. This prevents accidental
duplication of temporary integration columns and provides a clear starting point
for the final merge.


In [31]:
# Inspect the Spotify matching fields before integration

spotify_matching_columns = [
    column
    for column in spotify_df.columns
    if "normal" in column.lower()
]

print("SPOTIFY INTEGRATION KEY CHECK")
print("=" * 60)

print("\nNormalised columns currently available:")
for column in spotify_matching_columns:
    print(f"- {column}")

print("\nSpotify dataset shape:")
print(spotify_df.shape)

SPOTIFY INTEGRATION KEY CHECK

Normalised columns currently available:

Spotify dataset shape:
(4593, 27)


### 10.2 Create Spotify Integration Keys

To support the final dataset integration, normalised track and artist identifiers are created for the Spotify dataset. These fields provide consistent matching keys that can be used to attach the historical chart features generated in the previous stage.


In [32]:
# Create normalised integration keys for the Spotify dataset

spotify_integrated = spotify_df.copy()

spotify_integrated["track_normalised"] = (
    spotify_integrated["Track"]
    .astype(str)
    .str.strip()
    .str.lower()
)

spotify_integrated["artist_normalised"] = (
    spotify_integrated["Artist"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Spotify integration keys created.")
print("=" * 60)

print(f"Rows: {len(spotify_integrated):,}")
print(f"Columns: {spotify_integrated.shape[1]}")

print("\nIntegration key examples:")

spotify_integrated[
    [
        "Track",
        "Artist",
        "track_normalised",
        "artist_normalised"
    ]
].head(10)

Spotify integration keys created.
Rows: 4,593
Columns: 29

Integration key examples:


,Track,Artist,track_normalised,artist_normalised
0,MILLION DOLLAR BABY,Tommy Richman,million dollar baby,tommy richman
1,Not Like Us,Kendrick Lamar,not like us,kendrick lamar
2,i like the way you kiss me,Artemas,i like the way you kiss me,artemas
3,Flowers,Miley Cyrus,flowers,miley cyrus
4,Houdini,Eminem,houdini,eminem
5,Lovin On Me,Jack Harlow,lovin on me,jack harlow
6,Beautiful Things,Benson Boone,beautiful things,benson boone
7,Gata Only,FloyyMenor,gata only,floyymenor
8,Danza Kuduro - Cover,MUSIC LAB JPN,danza kuduro - cover,music lab jpn
9,BAND4BAND (feat. Lil Baby),Central Cee,band4band (feat. lil baby),central cee


### 10.3 Validate Historical Feature Matches

Before merging the historical chart features with the Spotify dataset, the normalised track and artist keys are compared between both datasets. This validation measures how many Spotify records can be reliably linked to the historical chart information and ensures that the integration does not unexpectedly duplicate records.


In [33]:
# Validate matches between Spotify records and historical chart features

spotify_keys = spotify_integrated[
    ["track_normalised", "artist_normalised"]
].drop_duplicates()

historical_keys = historical_features[
    ["track_normalised", "artist_normalised"]
].drop_duplicates()

# Find matching Spotify track-artist pairs
matched_keys = spotify_keys.merge(
    historical_keys,
    on=["track_normalised", "artist_normalised"],
    how="inner"
)

spotify_unique_pairs = len(spotify_keys)
historical_unique_pairs = len(historical_keys)
matched_pairs = len(matched_keys)

match_rate = (
    matched_pairs / spotify_unique_pairs * 100
    if spotify_unique_pairs > 0
    else 0
)

print("SPOTIFY ↔ HISTORICAL FEATURE MATCH VALIDATION")
print("=" * 60)

print(f"Unique Spotify track-artist pairs: {spotify_unique_pairs:,}")
print(f"Historical feature pairs: {historical_unique_pairs:,}")
print(f"Matched Spotify pairs: {matched_pairs:,}")
print(f"Spotify pair match rate: {match_rate:.2f}%")

print("\nHistorical feature key duplicates:")
print(
    historical_features.duplicated(
        subset=["track_normalised", "artist_normalised"]
    ).sum()
)

print("\nExample matched pairs:")

matched_keys.head(10)

SPOTIFY ↔ HISTORICAL FEATURE MATCH VALIDATION
Unique Spotify track-artist pairs: 4,480
Historical feature pairs: 2,178
Matched Spotify pairs: 2,178
Spotify pair match rate: 48.62%

Historical feature key duplicates:
0

Example matched pairs:


,track_normalised,artist_normalised
0,flowers,miley cyrus
1,lala,myke towers
2,as it was,harry styles
3,stay (with justin bieber),the kid laroi
4,baby shark,pinkfong
5,numb / encore,jay-z
6,dance monkey,tones and i
7,i'm good (blue),david guetta
8,if we ever broke up,mae stephens
9,despacito,luis fonsi


### 10.4 Integrate Historical Chart Features

The aggregated historical chart features are merged into the Spotify dataset using the normalised track and artist identifiers. A left join is used so that all Spotify records are retained, including tracks without corresponding historical chart observations. Historical feature fields therefore remain missing where no reliable historical match exists.


In [34]:
# Merge historical chart features into the Spotify dataset

spotify_with_history = spotify_integrated.merge(
    historical_features,
    on=["track_normalised", "artist_normalised"],
    how="left",
    validate="many_to_one"
)

print("HISTORICAL FEATURES INTEGRATED")
print("=" * 60)

print(f"Spotify rows before merge: {len(spotify_integrated):,}")
print(f"Rows after merge: {len(spotify_with_history):,}")
print(f"Columns before merge: {spotify_integrated.shape[1]}")
print(f"Columns after merge: {spotify_with_history.shape[1]}")

# Check whether the merge changed the number of Spotify records
row_difference = len(spotify_with_history) - len(spotify_integrated)

print(f"\nRow difference: {row_difference:,}")

# Determine how many Spotify rows received historical information
matched_rows = spotify_with_history["best_chart_position"].notna().sum()
unmatched_rows = spotify_with_history["best_chart_position"].isna().sum()

print("\nHistorical feature coverage:")
print(f"Rows with historical features: {matched_rows:,}")
print(f"Rows without historical features: {unmatched_rows:,}")
print(
    f"Coverage: "
    f"{matched_rows / len(spotify_with_history) * 100:.2f}%"
)

spotify_with_history.head()

HISTORICAL FEATURES INTEGRATED
Spotify rows before merge: 4,593
Rows after merge: 4,593
Columns before merge: 29
Columns after merge: 38

Row difference: 0

Historical feature coverage:
Rows with historical features: 2,235
Rows without historical features: 2,358
Coverage: 48.66%


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,artist_normalised,best_chart_position,average_chart_position,chart_observations,countries_charted,total_historical_streams,average_historical_streams,max_historical_streams,first_chart_date,last_chart_date
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,2024-04-26,QM24S2402528,1,725.4,3.904709e+08,30716.0,196631588.0,...,tommy richman,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
1,Not Like Us,Not Like Us,Kendrick Lamar,2024-05-04,USUG12400910,2,545.9,3.237039e+08,28113.0,174597137.0,...,kendrick lamar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
2,i like the way you kiss me,I like the way you kiss me,Artemas,2024-03-19,QZJ842400387,3,538.4,6.013093e+08,54331.0,211607669.0,...,artemas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT
3,Flowers,Flowers - Single,Miley Cyrus,2023-01-12,USSM12209777,4,444.9,2.031281e+09,269802.0,136569078.0,...,miley cyrus,1.0,19.7,873.0,74.0,1.658244e+09,1899477.9,115156896.0,2023-01-19,2023-04-06
4,Houdini,Houdini,Eminem,2024-05-31,USUG12403398,5,423.3,1.070349e+08,7223.0,151469874.0,...,eminem,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT


### 10.5 Prepare Artist Listener Integration

The listener dataset contains artist-level popularity information rather than individual track information. To integrate these features with the Spotify track dataset, artist names are normalised to create a consistent matching key. The uniqueness of the listener artist records is also inspected before performing the merge.


In [35]:
# Prepare listener dataset for artist-level integration

listeners_integrated = listeners_df.copy()

listeners_integrated["artist_normalised"] = (
    listeners_integrated["Artist"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("LISTENER INTEGRATION PREPARATION")
print("=" * 60)

print(f"Rows: {len(listeners_integrated):,}")
print(f"Columns: {listeners_integrated.shape[1]}")

print(f"\nUnique artists: {listeners_integrated['artist_normalised'].nunique():,}")

duplicate_artists = listeners_integrated.duplicated(
    subset=["artist_normalised"]
).sum()

print(f"Duplicate normalised artists: {duplicate_artists:,}")

print("\nExample listener integration keys:")

listeners_integrated[
    [
        "Artist",
        "artist_normalised",
        "Listeners",
        "Daily Trend",
        "Peak",
        "PkListeners"
    ]
].head(10)

LISTENER INTEGRATION PREPARATION
Rows: 2,500
Columns: 6

Unique artists: 2,500
Duplicate normalised artists: 0

Example listener integration keys:


,Artist,artist_normalised,Listeners,Daily Trend,Peak,PkListeners
0,The Weeknd,the weeknd,107592328,-138880,1,113034886
1,Taylor Swift,taylor swift,101003302,889,2,101003302
2,Ed Sheeran,ed sheeran,76475126,-68137,2,87934910
3,Dua Lipa,dua lipa,76421916,-71356,4,77778397
4,Bad Bunny,bad bunny,76162057,-199052,3,83950570
5,Rihanna,rihanna,75784389,116405,2,80958750
6,Drake,drake,75371611,-67013,6,76391086
7,Justin Bieber,justin bieber,72623228,34540,6,75467229
8,Billie Eilish,billie eilish,71793820,-72877,8,72368528
9,Miley Cyrus,miley cyrus,71277599,71089,2,84140935


### 10.6 Validate Spotify and Listener Artist Matches

The normalised artist identifiers from the Spotify and listener datasets are compared before integration. This validation determines how many Spotify artists have corresponding listener information and confirms that the listener dataset can be safely joined at the artist level.


In [36]:
# Validate artist matches between Spotify and listener datasets

spotify_artist_keys = (
    spotify_with_history[["artist_normalised"]]
    .drop_duplicates()
)

listener_artist_keys = (
    listeners_integrated[["artist_normalised"]]
    .drop_duplicates()
)

matched_artist_keys = spotify_artist_keys.merge(
    listener_artist_keys,
    on="artist_normalised",
    how="inner"
)

spotify_unique_artists = len(spotify_artist_keys)
listener_unique_artists = len(listener_artist_keys)
matched_artists = len(matched_artist_keys)

artist_match_rate = (
    matched_artists / spotify_unique_artists * 100
    if spotify_unique_artists > 0
    else 0
)

print("SPOTIFY ↔ LISTENER ARTIST MATCH VALIDATION")
print("=" * 60)

print(f"Unique Spotify artists: {spotify_unique_artists:,}")
print(f"Unique listener artists: {listener_unique_artists:,}")
print(f"Matched artists: {matched_artists:,}")
print(f"Spotify artist match rate: {artist_match_rate:.2f}%")

print("\nListener key duplicates:")
print(
    listeners_integrated.duplicated(
        subset=["artist_normalised"]
    ).sum()
)

print("\nExample matched artists:")

matched_artist_keys.head(15)

SPOTIFY ↔ LISTENER ARTIST MATCH VALIDATION
Unique Spotify artists: 1,997
Unique listener artists: 2,500
Matched artists: 1,003
Spotify artist match rate: 50.23%

Listener key duplicates:
0

Example matched artists:


,artist_normalised
0,kendrick lamar
1,miley cyrus
2,eminem
3,jack harlow
4,benson boone
5,central cee
6,post malone
7,teddy swims
8,billie eilish
9,future


### 10.7 Integrate Artist Listener Features

The artist-level listener features are merged with the Spotify and historical chart dataset using the normalised artist identifier. A left join is used to preserve every Spotify track, while listener information is added where a corresponding artist is available. The merge is validated as a many-to-one relationship because each artist has only one record in the listener dataset.


In [37]:
# Select listener features required for integration
listener_features = listeners_integrated[
    [
        "artist_normalised",
        "Listeners",
        "Daily Trend",
        "Peak",
        "PkListeners"
    ]
].copy()

# Merge listener features into the Spotify + historical dataset
integrated_df = spotify_with_history.merge(
    listener_features,
    on="artist_normalised",
    how="left",
    validate="many_to_one"
)

print("ARTIST LISTENER FEATURES INTEGRATED")
print("=" * 60)

print(f"Rows before merge: {len(spotify_with_history):,}")
print(f"Rows after merge: {len(integrated_df):,}")

print(f"Columns before merge: {spotify_with_history.shape[1]}")
print(f"Columns after merge: {integrated_df.shape[1]}")

row_difference = len(integrated_df) - len(spotify_with_history)

print(f"\nRow difference: {row_difference:,}")

# Listener feature coverage
matched_listener_rows = integrated_df["Listeners"].notna().sum()
unmatched_listener_rows = integrated_df["Listeners"].isna().sum()

listener_coverage = (
    matched_listener_rows / len(integrated_df) * 100
)

print("\nListener feature coverage:")
print(f"Rows with listener features: {matched_listener_rows:,}")
print(f"Rows without listener features: {unmatched_listener_rows:,}")
print(f"Coverage: {listener_coverage:.2f}%")

integrated_df[
    [
        "Track",
        "Artist",
        "Spotify Streams",
        "best_chart_position",
        "Listeners",
        "Daily Trend",
        "Peak",
        "PkListeners"
    ]
].head(10)

ARTIST LISTENER FEATURES INTEGRATED
Rows before merge: 4,593
Rows after merge: 4,593
Columns before merge: 38
Columns after merge: 42

Row difference: 0

Listener feature coverage:
Rows with listener features: 3,389
Rows without listener features: 1,204
Coverage: 73.79%


,Track,Artist,Spotify Streams,best_chart_position,Listeners,Daily Trend,Peak,PkListeners
0,MILLION DOLLAR BABY,Tommy Richman,3.904709e+08,NaN,NaN,NaN,NaN,NaN
1,Not Like Us,Kendrick Lamar,3.237039e+08,NaN,47391930.0,-8503.0,33.0,54045549.0
2,i like the way you kiss me,Artemas,6.013093e+08,NaN,NaN,NaN,NaN,NaN
3,Flowers,Miley Cyrus,2.031281e+09,1.0,71277599.0,71089.0,2.0,84140935.0
4,Houdini,Eminem,1.070349e+08,NaN,64022485.0,-52362.0,11.0,68591390.0
5,Lovin On Me,Jack Harlow,6.706654e+08,NaN,26620407.0,-14428.0,103.0,32198313.0
6,Beautiful Things,Benson Boone,9.001588e+08,NaN,14274669.0,-10909.0,488.0,14307536.0
7,Gata Only,FloyyMenor,6.750792e+08,NaN,NaN,NaN,NaN,NaN
8,Danza Kuduro - Cover,MUSIC LAB JPN,1.653018e+09,NaN,NaN,NaN,NaN,NaN
9,BAND4BAND (feat. Lil Baby),Central Cee,9.067657e+07,NaN,32987690.0,-33056.0,89.0,35154343.0


### 10.8 Final Integrated Dataset Validation

The fully integrated dataset is validated before it is saved for later analysis and machine learning. The checks confirm that the original Spotify row structure has been preserved, examine duplicate records and missing values, and measure the coverage of the historical chart and artist listener features. This ensures that the integration process has not unintentionally introduced or removed Spotify observations.


In [40]:
# Final validation of the integrated dataset

print("FINAL INTEGRATED DATASET VALIDATION")
print("=" * 65)

# --------------------------------------------------
# 1. Dataset structure
# --------------------------------------------------

print("\nDATASET STRUCTURE")
print("-" * 65)

print(f"Original Spotify rows: {len(spotify_df):,}")
print(f"Integrated rows:       {len(integrated_df):,}")
print(f"Integrated columns:    {integrated_df.shape[1]}")

row_structure_preserved = (
    len(spotify_df) == len(integrated_df)
)

print(
    f"Row structure preserved: "
    f"{row_structure_preserved}"
)

# --------------------------------------------------
# 2. Original Spotify identity check
# --------------------------------------------------

print("\nSPOTIFY RECORD CHECK")
print("-" * 65)

# spotify_integrated contains the temporary normalised keys
original_pairs = set(
    zip(
        spotify_integrated["track_normalised"],
        spotify_integrated["artist_normalised"]
    )
)

integrated_pairs = set(
    zip(
        integrated_df["track_normalised"],
        integrated_df["artist_normalised"]
    )
)

print(
    f"Original unique track-artist pairs:   "
    f"{len(original_pairs):,}"
)

print(
    f"Integrated unique track-artist pairs: "
    f"{len(integrated_pairs):,}"
)

track_identity_preserved = (
    original_pairs == integrated_pairs
)

print(
    f"Track-artist identity preserved: "
    f"{track_identity_preserved}"
)

# --------------------------------------------------
# 3. Duplicate check
# --------------------------------------------------

print("\nDUPLICATE CHECK")
print("-" * 65)

duplicate_rows = (
    integrated_df
    .duplicated()
    .sum()
)

duplicate_pairs = (
    integrated_df
    .duplicated(
        subset=[
            "track_normalised",
            "artist_normalised"
        ]
    )
    .sum()
)

print(
    f"Exact duplicate rows: "
    f"{duplicate_rows:,}"
)

print(
    f"Duplicate normalised track-artist pairs: "
    f"{duplicate_pairs:,}"
)

# --------------------------------------------------
# 4. Historical feature coverage
# --------------------------------------------------

print("\nHISTORICAL CHART COVERAGE")
print("-" * 65)

history_rows = (
    integrated_df["best_chart_position"]
    .notna()
    .sum()
)

history_missing = (
    integrated_df["best_chart_position"]
    .isna()
    .sum()
)

history_coverage = (
    history_rows
    / len(integrated_df)
    * 100
)

print(
    f"Rows with historical features:    "
    f"{history_rows:,}"
)

print(
    f"Rows without historical features: "
    f"{history_missing:,}"
)

print(
    f"Coverage: "
    f"{history_coverage:.2f}%"
)

# --------------------------------------------------
# 5. Listener feature coverage
# --------------------------------------------------

print("\nLISTENER FEATURE COVERAGE")
print("-" * 65)

listener_rows = (
    integrated_df["Listeners"]
    .notna()
    .sum()
)

listener_missing = (
    integrated_df["Listeners"]
    .isna()
    .sum()
)

listener_coverage = (
    listener_rows
    / len(integrated_df)
    * 100
)

print(
    f"Rows with listener features:    "
    f"{listener_rows:,}"
)

print(
    f"Rows without listener features: "
    f"{listener_missing:,}"
)

print(
    f"Coverage: "
    f"{listener_coverage:.2f}%"
)

# --------------------------------------------------
# 6. Combined enrichment coverage
# --------------------------------------------------

print("\nCOMBINED FEATURE COVERAGE")
print("-" * 65)

both_features = (
    integrated_df["best_chart_position"].notna()
    &
    integrated_df["Listeners"].notna()
).sum()

history_only = (
    integrated_df["best_chart_position"].notna()
    &
    integrated_df["Listeners"].isna()
).sum()

listener_only = (
    integrated_df["best_chart_position"].isna()
    &
    integrated_df["Listeners"].notna()
).sum()

neither = (
    integrated_df["best_chart_position"].isna()
    &
    integrated_df["Listeners"].isna()
).sum()

at_least_one = (
    len(integrated_df) - neither
)

at_least_one_percentage = (
    at_least_one
    / len(integrated_df)
    * 100
)

print(
    f"Historical + listener features: "
    f"{both_features:,}"
)

print(
    f"Historical only:                "
    f"{history_only:,}"
)

print(
    f"Listener only:                  "
    f"{listener_only:,}"
)

print(
    f"Neither enrichment source:      "
    f"{neither:,}"
)

print(
    f"\nRows with at least one enrichment source: "
    f"{at_least_one:,} "
    f"({at_least_one_percentage:.2f}%)"
)

# --------------------------------------------------
# 7. Missing values overview
# --------------------------------------------------

print("\nMISSING VALUE OVERVIEW")
print("-" * 65)

missing_summary = (
    integrated_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = (
    missing_summary[
        missing_summary > 0
    ]
)

if len(missing_summary) > 0:
    print(missing_summary)
else:
    print("No missing values found.")

# --------------------------------------------------
# 8. Final validation summary
# --------------------------------------------------

print("\nFINAL VALIDATION SUMMARY")
print("-" * 65)

if (
    row_structure_preserved
    and track_identity_preserved
):
    print("✓ Original Spotify row structure preserved.")
    print("✓ Track-artist identities preserved.")
    print("✓ Historical chart integration validated.")
    print("✓ Artist listener integration validated.")
    print("✓ Integrated dataset is ready for the next stage.")
else:
    print(
        "⚠ Validation detected a structural inconsistency."
    )

print("\nValidation complete.")

FINAL INTEGRATED DATASET VALIDATION

DATASET STRUCTURE
-----------------------------------------------------------------
Original Spotify rows: 4,593
Integrated rows:       4,593
Integrated columns:    42
Row structure preserved: True

SPOTIFY RECORD CHECK
-----------------------------------------------------------------
Original unique track-artist pairs:   4,480
Integrated unique track-artist pairs: 4,480
Track-artist identity preserved: True

DUPLICATE CHECK
-----------------------------------------------------------------
Exact duplicate rows: 0
Duplicate normalised track-artist pairs: 113

HISTORICAL CHART COVERAGE
-----------------------------------------------------------------
Rows with historical features:    2,235
Rows without historical features: 2,358
Coverage: 48.66%

LISTENER FEATURE COVERAGE
-----------------------------------------------------------------
Rows with listener features:    3,389
Rows without listener features: 1,204
Coverage: 73.79%

COMBINED FEATURE COVER

### 10.9 Save the Integrated Dataset

Following successful validation, the final integrated dataset is saved to the
dedicated `data/integrated/` directory. Keeping integrated outputs separate from
the cleaned datasets in `data/processed/` makes the project structure clearer
and prevents generated integration outputs from being confused with cleaned
source datasets.

The saved file becomes the reusable input for exploratory data analysis,
feature engineering and subsequent machine-learning stages.


In [ ]:
from pathlib import Path

# Create the integrated data directory if it does not already exist
integrated_dir = Path("../data/integrated")
integrated_dir.mkdir(parents=True, exist_ok=True)

# Define the final integrated dataset path
integrated_file = integrated_dir / "spotify_integrated.csv"

# Save the integrated dataset
integrated_df.to_csv(integrated_file, index=False)

# Display save information
file_size_mb = integrated_file.stat().st_size / (1024 * 1024)

print("INTEGRATED DATASET SAVED")
print("=" * 65)
print(f"File: {integrated_file}")
print(f"Rows saved: {len(integrated_df):,}")
print(f"Columns saved: {integrated_df.shape[1]}")
print(f"File size: {file_size_mb:.2f} MB")
print("\nData integration stage completed successfully.")


### 10.10 Reload and Verify the Saved Dataset

As a final integrity check, the saved integrated dataset is reloaded from the
`data/integrated/` directory and compared with the in-memory dataset. This
confirms that the export completed successfully and that the expected number of
rows and columns was preserved.


In [ ]:
# Reload the saved integrated dataset
reloaded_integrated_df = pd.read_csv(integrated_file)

print("SAVED DATASET VERIFICATION")
print("=" * 65)

print(f"Expected shape: {integrated_df.shape}")
print(f"Reloaded shape: {reloaded_integrated_df.shape}")

shape_matches = reloaded_integrated_df.shape == integrated_df.shape

print(f"Shape preserved: {shape_matches}")
print(f"Rows preserved: {len(reloaded_integrated_df) == len(integrated_df)}")
print(
    f"Columns preserved: "
    f"{reloaded_integrated_df.shape[1] == integrated_df.shape[1]}"
)

if shape_matches:
    print("\nSaved integrated dataset verified successfully.")
else:
    print("\nWarning: the reloaded dataset shape does not match the in-memory dataset.")


## 11. Integration Summary

The three cleaned PMIP datasets were successfully connected using track-level
and artist-level matching strategies.

Spotify 2024 records were matched with historical Charts data using normalised
track and artist identifiers. Historical chart observations were aggregated
into compact performance features before being merged with the Spotify dataset.
Artist-level listener statistics were then integrated using normalised artist
identifiers.

The final integration preserves the original Spotify row structure while adding
historical chart-performance and listener-level features where reliable matches
are available.

The completed integrated dataset is saved as:

`data/integrated/spotify_integrated.csv`

This file is now ready for exploratory data analysis and visualisation.
